# Brain Tumor MRI Classification with Neural Networks and Deep Learning

**Course:** MLAI 504 – Neural Networks and Deep Learning (2026–2027)  
**Author:** Karam Jammal & Yorgo Bou Fayad
**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) (Kaggle, CC0: Public Domain)

## Project goal

Build a neural network that classifies a brain MRI slice into one of four classes: **glioma**, **meningioma**, **pituitary tumor** or **no tumor**. The project follows a systematic approach: data exploration and cleaning, step-by-step model creation, hyperparameter tuning and transfer learning, with every decision explained.

> ⚠️ **Disclaimer:** this is an academic project. The model is not a medical device and must not be used for diagnosis.

## Table of contents

0. Environment setup
1. Dataset selection and description
2. Data exploration and preprocessing
3. Step-by-step model creation
4. Hyperparameter tuning
5. Transfer learning
6. Integration of course topics
7. Evaluation and interpretation
8. Final results and conclusions
9. References

---
# 0. Environment setup

The notebook uses **PyTorch** and **torchvision**. This section installs the two libraries that are not always preinstalled, imports everything, fixes the random seeds and records the library versions so the results can be reproduced. Models are trained on a GPU (Google Colab) when one is available.

In [ ]:
# Install the libraries that are not always preinstalled (does nothing if they are already installed)
%pip install -q imagehash kagglehub

In [ ]:
import os
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cv2
import imagehash
import kagglehub
import torch
import torchvision

# Reproducibility: fix the Python, NumPy and PyTorch random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Record the environment
versions = {
    "Python": platform.python_version(),
    "PyTorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "OpenCV": cv2.__version__,
    "ImageHash": imagehash.__version__,
    "kagglehub": kagglehub.__version__,
}
for name, version in versions.items():
    print(f"{name:<14}: {version}")

# Train on the GPU when one is available, otherwise on the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

**Download the dataset.** `kagglehub` downloads the dataset from Kaggle the first time and reuses the local copy afterwards. The folder tree below confirms the data is in place and gives a first image count per class.

In [ ]:
# Download the dataset (only the first time; later runs reuse the local copy)
DATA_DIR = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset location:", DATA_DIR, "\n")

# Print the folder tree with the number of images in each folder
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
for root, dirs, files in sorted(os.walk(DATA_DIR)):
    depth = root[len(DATA_DIR):].count(os.sep)
    n_images = sum(f.lower().endswith(IMAGE_EXTENSIONS) for f in files)
    label = f"  ({n_images} images)" if n_images else ""
    print(f"{'    ' * depth}{os.path.basename(root)}/{label}")

---
# 1. Dataset selection and description

## Source
We use the **Brain Tumor MRI Dataset** published on Kaggle by Masoud Nickparvar (version 2, CC0: Public Domain). It combines three public datasets: **Figshare**, **SARTAJ** and **Br35H**. The "no tumor" images come from Br35H, and the author replaced SARTAJ's glioma images, which were found to be mislabeled, with images from Figshare.

## Why this dataset
Brain tumors are diagnosed mainly from MRI scans, and the tumor type matters because each type is treated differently. Reading scans takes expert time, so a model that classifies them automatically could support radiologists, for example by flagging the scans that need attention first. The dataset also fits this project well:
- it is an image dataset, which suits CNNs and transfer learning with pretrained models such as ResNet50;
- it is a manageable size (7,200 images, about 150 MB), so models train in minutes on a GPU;
- it has real data-quality problems to solve (different image sizes, black borders, and duplicate images across the training and test sets), which are handled in Section 2.

## Size, features and target
- **Size:** 7,200 JPEG images, provided in a `Training` folder (5,600) and a `Testing` folder (1,600).
- **Features:** the pixel values of each MRI slice. The scans are grayscale and come in different resolutions, so they are resized to a common size during preprocessing.
- **Target variable:** the tumor class, given by the folder name, with four categories:

| Class | Description |
|---|---|
| **glioma** | Tumor that grows from the glial (supporting) cells of the brain; can be slow-growing or aggressive |
| **meningioma** | Tumor of the meninges, the membranes around the brain; usually benign |
| **pituitary** | Tumor of the pituitary gland at the base of the brain; usually benign, but can affect hormones and vision |
| **notumor** | Healthy brain scan with no tumor |

**Image index.** We build a table with one row per image (file path, class and the folder it came from). It is used for counting here and for all preprocessing in Section 2.

In [ ]:
from pathlib import Path

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]

# One row per image: file path, class label and the folder it came from
records = []
for split in ["Training", "Testing"]:
    for label in CLASSES:
        for path in sorted((Path(DATA_DIR) / split / label).iterdir()):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({"path": str(path), "label": label, "original_split": split})

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")

# Number of images per class in each folder
counts = pd.crosstab(df["label"], df["original_split"], margins=True, margins_name="Total")
counts[["Training", "Testing", "Total"]]

**Sample images.** Four random images per class, with each image's size in pixels (width × height).

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(10, 10))

for row, label in enumerate(CLASSES):
    samples = df[df["label"] == label].sample(4, random_state=SEED)
    for col, path in enumerate(samples["path"]):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        ax = axes[row, col]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"{label}\n{img.shape[1]}×{img.shape[0]}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

**Observations**
- The four classes are perfectly balanced in the original dataset (1,800 images each). This changes after duplicate removal (Section 2.2).
- Image sizes vary between images, so all images must be resized to one common size before training.
- Many scans have black borders around the head, which carry no information and can be cropped.
- The scans show different views of the head (from above, from the front and from the side) and different contrast levels, because the images come from three different sources.
- The "notumor" images come from a different source (Br35H) and look different from the tumor images (other sizes and contrast styles). A model could learn these differences instead of the tumor itself, so we keep this in mind when interpreting the results.

---
# 2. Data exploration and preprocessing

Goal: a clean, leakage-free dataset ready for training, in five steps:
1. Integrity check and image sizes
2. Duplicate removal
3. Train / validation / test split
4. Cropping and resizing
5. Pixel scaling and data augmentation

## 2.1 Integrity check and image sizes
We open every image and record its size and colour mode. Images that can't be opened are the image equivalent of missing values and would be removed. We also compute a **perceptual hash** for each image: a short fingerprint that stays the same when an image is resized or re-saved, used in 2.2 to find duplicates.

In [ ]:
from PIL import Image

def inspect_image(path):
    """Return the image's size, colour mode and perceptual hash (or readable=False if it can't be opened)."""
    try:
        with Image.open(path) as img:
            img.load()
            return {"readable": True, "width": img.width, "height": img.height,
                    "mode": img.mode, "phash": str(imagehash.phash(img))}
    except Exception:
        return {"readable": False, "width": None, "height": None, "mode": None, "phash": None}

info = pd.DataFrame([inspect_image(p) for p in df["path"]])
df = pd.concat([df[["path", "label", "original_split"]], info], axis=1)

print("Unreadable images:", (~df["readable"]).sum())
print("Missing labels:", df["label"].isna().sum())
print("Colour modes:", df["mode"].value_counts().to_dict())

# Keep only images that could be opened
df = df[df["readable"]].reset_index(drop=True)

In [ ]:
df["size"] = df["width"].astype(str) + "×" + df["height"].astype(str)
print("Different image sizes:", df["size"].nunique())
print("\nMost common sizes:")
print(df["size"].value_counts().head(5).to_string())
print("\nWidth and height per class:")
df.groupby("label")[["width", "height"]].agg(["min", "median", "max"])

**Observations**
- No unreadable images and no missing labels were found.
- The images are stored in several colour modes (4,129 RGB, 3,067 grayscale, 3 RGBA and 1 palette image), even though all scans are grayscale, so every image is loaded as grayscale (1 channel).
- There are 447 different image sizes, from 150 to 1,375 pixels wide. Small and large images are kept, because every image is resized to the same size in 2.4.
- The "notumor" images are mostly smaller than the tumor images (median 228 × 243 vs 512 × 512 pixels), which confirms they come from a different source.

## 2.2 Duplicates and data leakage
Images with the same hash are copies of each other. A test image with a copy in the training folder is "leaked": the model would be tested on an image it already saw during training, which inflates the test score. We keep one copy of each image, and remove copies that carry different labels, since we can't know which label is correct.

In [ ]:
n_with_copy = df.duplicated("phash", keep=False).sum()
print(f"Images that have at least one copy: {n_with_copy}")

# Leakage: test images whose copy is in the training folder
train_hashes = set(df.loc[df["original_split"] == "Training", "phash"])
leaked = df[(df["original_split"] == "Testing") & df["phash"].isin(train_hashes)]
n_test = (df["original_split"] == "Testing").sum()
print(f"Test images with a copy in Training: {len(leaked)} of {n_test} ({len(leaked) / n_test:.1%})")

# Copies with different labels can't be trusted, so they are removed entirely
labels_per_hash = df.groupby("phash")["label"].nunique()
conflicting = labels_per_hash[labels_per_hash > 1].index
print(f"Copies with conflicting labels: {len(conflicting)} groups")

# Keep one copy of every remaining image
clean = df[~df["phash"].isin(conflicting)].drop_duplicates("phash").reset_index(drop=True)
print(f"\nImages: {len(df)} → {len(clean)} after removing duplicates")
print(clean["label"].value_counts().to_string())

In [ ]:
# Leaked test images (top) next to their copy in the training folder (bottom)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for col, (_, test_img) in enumerate(leaked.head(4).iterrows()):
    train_img = df[(df["phash"] == test_img["phash"]) & (df["original_split"] == "Training")].iloc[0]
    for row, img in enumerate([test_img, train_img]):
        axes[row, col].imshow(cv2.imread(img["path"], cv2.IMREAD_GRAYSCALE), cmap="gray")
        axes[row, col].set_title(f"{img['original_split']}: {img['label']}", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

**Observations**
- 370 of the 1,600 original test images (23.1%) have an identical copy in the training folder. Testing on the original split would overestimate the model's accuracy, which explains the very high scores often reported for this dataset.
- In total, 1,580 images had at least one copy, and 1 group of copies had conflicting labels and was removed.
- After removing duplicates, 6,211 of 7,200 images remain. The "notumor" class lost the most images (1,800 → 1,023), so the classes are now moderately imbalanced (1,023 to 1,774 per class). We compensate with class weights during training (Section 3).

## 2.3 Train / validation / test split
We ignore the original Training/Testing folders and split the clean images **70% / 15% / 15%**, stratified so each set keeps the same class proportions. Because duplicates were removed first, no image appears in two sets.
- **Train:** used to fit the models
- **Validation:** used to compare models and choose hyperparameters
- **Test:** used only once, at the very end (Section 7), for the final unbiased score

The split is saved to `data/split_manifest.csv` so it can be reproduced exactly.

In [ ]:
from sklearn.model_selection import train_test_split

train_idx, temp_idx = train_test_split(clean.index, test_size=0.30, stratify=clean["label"], random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=clean.loc[temp_idx, "label"], random_state=SEED)

clean["split"] = "train"
clean.loc[val_idx, "split"] = "val"
clean.loc[test_idx, "split"] = "test"
print(pd.crosstab(clean["label"], clean["split"], margins=True, margins_name="Total")[["train", "val", "test", "Total"]])

# Save the split (file paths relative to the dataset folder, so it works on any computer)
clean["file"] = [Path(p).relative_to(DATA_DIR).as_posix() for p in clean["path"]]
os.makedirs("../data", exist_ok=True)
clean[["file", "label", "split"]].to_csv("../data/split_manifest.csv", index=False)

## 2.4 Cropping and resizing
Each image is cropped to the box around the head, removing the black borders that carry no information, then resized to **224 × 224** pixels, the input size used by most pretrained models (Section 5). Images stay grayscale (1 channel) and are stored channels-first, with shape (1, 224, 224), which is the layout PyTorch expects.

In [ ]:
IMG_SIZE = 224

def crop_and_resize(path, size=IMG_SIZE):
    """Crop the image to the head (removing the black borders) and resize it to size × size."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    # Mark the pixels brighter than the background, ignoring small specks of noise
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    _, mask = cv2.threshold(blurred, 45, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    # Crop to the box that contains all the bright pixels (the head)
    ys, xs = np.nonzero(mask)
    if len(xs) > 0:
        img = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

# One example per class: original (top) and after cropping + resizing (bottom)
examples = clean.groupby("label").sample(1, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for col, (_, item) in enumerate(examples.iterrows()):
    axes[0, col].imshow(cv2.imread(item["path"], cv2.IMREAD_GRAYSCALE), cmap="gray")
    axes[0, col].set_title(f"{item['label']} (original)", fontsize=9)
    axes[1, col].imshow(crop_and_resize(item["path"]), cmap="gray")
    axes[1, col].set_title("cropped + resized", fontsize=9)
    for row in range(2):
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def load_split(split):
    """Crop and resize all images of one split; return images (n, 1, 224, 224) and integer labels."""
    part = clean[clean["split"] == split]
    X = np.stack([crop_and_resize(p) for p in part["path"]])[:, np.newaxis]
    y = part["label"].map(CLASSES.index).to_numpy()
    return X, y

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)
print("Pixel values:", X_train.min(), "to", X_train.max(), "| type:", X_train.dtype)

## 2.5 Pixel scaling and data augmentation
- **Scaling:** pixel values (0–255) are scaled to 0–1 each time an image is loaded for training or evaluation (Section 2.6).
- **Augmentation:** during training, each image is randomly flipped left–right, rotated by up to ±18° and zoomed by up to 10%, so the model sees slightly different versions every epoch and generalises better. Left–right flips are realistic because tumors occur on both sides of the brain; upside-down flips are not, so they are not used. Augmentation is applied to the training images only.

In [ ]:
from torchvision.transforms import v2

# Random left–right flip, rotation up to ±18° and zoom up to ±10% (empty areas filled with black)
train_transform = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomAffine(degrees=18, scale=(0.9, 1.1)),
])

# One training image (scaled to 0–1) and five random augmented versions of it
sample = torch.from_numpy(X_train[0]).float() / 255
plt.figure(figsize=(12, 3))
for i in range(6):
    image = sample if i == 0 else train_transform(sample)
    plt.subplot(1, 6, i + 1)
    plt.imshow(image[0].numpy(), cmap="gray")
    plt.title("original" if i == 0 else f"augmented {i}", fontsize=9)
    plt.axis("off")
plt.show()

## 2.6 Datasets and data loaders
A PyTorch `Dataset` serves one image at a time: it scales the pixels to 0–1 and, for the training set only, applies the augmentation. A `DataLoader` groups the images into batches of 32, shuffled for training.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MRIDataset(Dataset):
    """Serves one image at a time as a float tensor scaled to 0–1, with optional augmentation."""
    def __init__(self, X, y, transform=None):
        self.X = torch.from_numpy(X)                # (n, 1, 224, 224), uint8
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        image = self.X[i].float() / 255             # scale pixels from 0–255 to 0–1
        if self.transform:
            image = self.transform(image)
        return image, self.y[i]

BATCH_SIZE = 32
train_loader = DataLoader(MRIDataset(X_train, y_train, transform=train_transform), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MRIDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(MRIDataset(X_test, y_test), batch_size=BATCH_SIZE)

images, labels = next(iter(train_loader))
print("One training batch:", tuple(images.shape), images.dtype, "| labels:", tuple(labels.shape))
print(f"Pixel values in the batch: {images.min():.2f} to {images.max():.2f}")

**Preprocessing summary**

| Step | Result |
|---|---|
| Integrity check | 0 unreadable images, 0 missing labels |
| Duplicates | 7,200 → 6,211 images (370 test images had a copy in training) |
| Split | 4,347 train / 932 validation / 932 test, stratified |
| Cropping and resizing | all images 224 × 224, grayscale, shape (1, 224, 224) |
| Scaling | 0–255 → 0–1 when each image is loaded |
| Augmentation | flip, rotation, zoom (training only) |
| Data loaders | batches of 32, shuffled for training |